# Orbital Entanglement Chord Diagram — 250 Synthetic Orbitals

Demonstrates the `Entanglement` JS widget for an orbital-entanglement use case on a large system with
synthetic single-orbital entropies and mutual information.  Five highly entangled clusters are
highlighted with distinct outline colours and grouped together on the ring.

## 1 — Build synthetic data

In [ ]:
import numpy as np

rng = np.random.default_rng(42)

N = 250        # total orbitals
N_CORE = 50    # strongly-coupled subset to highlight

# ── Single-orbital entropies ─────────────────────────────────────────
# Three regimes interleaved across the orbital indices:
#   • ~50 "core" orbitals with high entropy (near ln 4)
#   • ~60 "medium" orbitals with moderate entropy
#   • ~140 "spectator" orbitals with a long decaying tail
s1 = np.zeros(N)

all_indices = rng.permutation(N)
core_idx = np.sort(all_indices[:N_CORE])
medium_idx = np.sort(all_indices[N_CORE : N_CORE + 60])
tail_idx = np.sort(all_indices[N_CORE + 60 :])

s1[core_idx] = rng.beta(2.5, 1.2, len(core_idx)) * np.log(4.0)
s1[medium_idx] = rng.beta(1.8, 3.0, len(medium_idx)) * np.log(4.0) * 0.5
s1[tail_idx] = np.sort(rng.exponential(0.03, len(tail_idx)))[::-1]

# ── Mutual information matrix ──────────────────────────────────────
mi = np.zeros((N, N))

# 1) Five intra-core clusters of ~10 orbitals each with strong MI
cluster_size = len(core_idx) // 5
clusters = []
for k in range(5):
    cl = core_idx[k * cluster_size : (k + 1) * cluster_size]
    clusters.append(cl)
    for ii, i in enumerate(cl):
        for j in cl[ii + 1 :]:
            val = rng.beta(3, 1.5) * np.log(16.0) * 0.55
            mi[i, j] = mi[j, i] = val

# 2) Sparse inter-cluster core links
for ii, i in enumerate(core_idx):
    for j in core_idx[ii + 1 :]:
        if mi[i, j] == 0 and rng.random() < 0.15:
            val = rng.exponential(0.08)
            mi[i, j] = mi[j, i] = val

# 3) Medium orbitals: moderate MI to a few core and medium neighbours
for i in medium_idx:
    n_core_links = rng.integers(1, 4)
    targets = rng.choice(core_idx, size=n_core_links, replace=False)
    for j in targets:
        val = rng.exponential(0.12)
        mi[i, j] = mi[j, i] = val
    n_med_links = rng.integers(0, 3)
    others = rng.choice(
        medium_idx[medium_idx != i],
        size=min(n_med_links, len(medium_idx) - 1),
        replace=False,
    )
    for j in others:
        if mi[i, j] == 0:
            val = rng.exponential(0.06)
            mi[i, j] = mi[j, i] = val

# 4) Tail orbitals: very sparse, weak links to core or medium
for i in tail_idx:
    if rng.random() < 0.12:
        pool = np.concatenate([core_idx, medium_idx])
        j = rng.choice(pool)
        val = rng.exponential(0.02)
        mi[i, j] = mi[j, i] = val

np.fill_diagonal(mi, 0.0)

# All five highly-entangled clusters
region_a, region_b, region_c, region_d, region_e = clusters

print(f"{N} orbitals, 5 entangled clusters built ({cluster_size} orbitals each)")
for name, region in zip("ABCDE", clusters):
    print(f"  Cluster {name}: {region.tolist()}")
print(f"s1 range: {s1.min():.4f} – {s1.max():.4f}")
print(f"MI range: {mi[mi > 0].min():.4f} – {mi.max():.4f}")
print(f"Non-zero MI pairs: {(mi > 0).sum() // 2}")

## 2 — Display the interactive widget

The `Entanglement` widget renders an orbital-entanglement chord diagram
directly in the notebook output. Arc length encodes single-orbital
entropy; chord thickness encodes mutual information. Five highly
entangled regions (clusters of strongly-coupled orbitals) are each
outlined in a different colour and, when the grouping toggle is active,
placed adjacent on the ring.

### Visual options

The widget constructor accepts keyword arguments that tune the diagram
appearance.  This sample uses:

| Option | Value | Effect |
|---|---|---|
| `gap_deg` | `0.6` | Narrow gap (degrees) between arcs — tight packing for 250 orbitals |
| `arc_width` | `0.05` | Thinner arcs (fraction of radius; default `0.08`) |
| `mi_threshold` | `0.01` | Hide chords with mutual information below 0.01 to reduce clutter |
| `group_selected` | `True` | Reorder arcs so each group's members sit adjacent on the ring |
| `width` / `height` | `800` / `880` | Larger viewport (default 600 × 660) |

Other options: `radius`, `line_scale`, `s1_vmax`, `mi_vmax`,
`selection_color`, `selection_linewidth`, `group_colors`, `title`.
See `help(Entanglement)` for the full list.

In [ ]:
from qsharp_widgets import Entanglement

widget = Entanglement(
    s1_entropies=s1.tolist(),
    mutual_information=mi.tolist(),
    labels=[str(i) for i in range(N)],
    groups={
        "Region A": region_a.tolist(),
        "Region B": region_b.tolist(),
        "Region C": region_c.tolist(),
        "Region D": region_d.tolist(),
        "Region E": region_e.tolist(),
    },
    title=f"Synthetic Orbital Entanglement — {N} orbitals (5 entangled regions)",
    group_selected=True,
    gap_deg=0.6,
    arc_width=0.05,
    mi_threshold=0.01,
    width=800,
    height=880,
)
widget